# Previsão ECMWF para Minas Gerais
Este notebook demonstra como baixar dados da ECMWF e visualizar mapas com a localização da cidade de Viçosa (MG).


## Instalação de dependências
Pode ser necessário instalar alguns pacotes extras como `cdsapi`, `xarray`, `cartopy` e `ipywidgets`.
```python
!pip install cdsapi xarray cartopy ipywidgets
```


In [ ]:
import ipywidgets as widgets
from IPython.display import display

opcoes = ['Precipitação acumulada', 'Temperatura']
variavel = widgets.Dropdown(options=opcoes, description='Variável:')
display(variavel)


In [ ]:
import cdsapi

def baixar_dados(var):
    codigo = 'total_precipitation' if var == 'Precipitação acumulada' else '2m_temperature'
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': codigo,
            'year': '2023',
            'month': '06',
            'day': '01',
            'time': '00:00',
            'area': [-15, -48, -22.5, -40],  # recorte aproximado de Minas
            'format': 'netcdf'
        },
        f'{codigo}.nc'
    )
    print('Arquivo salvo:', f'{codigo}.nc')


In [ ]:
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        baixar_dados(change['new'])

variavel.observe(on_change)


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import natural_earth

def plotar(var):
    codigo = 'total_precipitation' if var == 'Precipitação acumulada' else '2m_temperature'
    ds = xr.open_dataset(f'{codigo}.nc')
    data = ds[codigo].isel(time=0)

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    data.plot(ax=ax, cmap='viridis', cbar_kwargs={'label': var})

    estados = cfeature.NaturalEarthFeature('cultural', 'admin_1_states_provinces', '10m', edgecolor='gray', facecolor='none')
    ax.add_feature(estados)
    ax.coastlines(resolution='10m')

    # marcação de Viçosa
    lat_vicosa, lon_vicosa = -20.7554, -42.8786
    ax.plot(lon_vicosa, lat_vicosa, 'ro', markersize=5, transform=ccrs.PlateCarree())
    ax.text(lon_vicosa + 0.1, lat_vicosa + 0.1, 'Viçosa', transform=ccrs.PlateCarree())

    ax.set_title(f'{var} - Minas Gerais')
    plt.show()

plotar(variavel.value)
